# 6. Interactive Visual Analytics — Folium
Run the data-wrangling notebook first. The map displays launchpad locations returned by the API and a popup summary based on available launch records.

In [ ]:
import pandas as pd
from pathlib import Path
import folium
from folium.plugins import MarkerCluster

ROOT = Path("..")
df = pd.read_csv(ROOT / "data" / "processed" / "spacex_falcon9_launches.csv")
df = df.dropna(subset=["latitude","longitude"])
if df.empty:
    raise ValueError("No launch site coordinates found in processed dataset.")
center = [df["latitude"].mean(), df["longitude"].mean()]
m = folium.Map(location=center, zoom_start=4, tiles="OpenStreetMap")
cluster = MarkerCluster().add_to(m)
summary = df.groupby(["launch_site","latitude","longitude"], dropna=False).agg(
    launch_records=("flight_number","count"),
    mission_successes=("mission_success", lambda s: int((s == True).sum()))
).reset_index()
for _, row in summary.iterrows():
    popup = (f"<b>{row['launch_site']}</b><br>Records: {row['launch_records']}<br>"
             f"Mission successes: {row['mission_successes']}")
    folium.Marker([row["latitude"], row["longitude"]], popup=popup, tooltip=str(row["launch_site"])).add_to(cluster)
m

In [ ]:
out = ROOT / "outputs" / "figures" / "spacex_launch_sites_map.html"
out.parent.mkdir(parents=True, exist_ok=True)
m.save(out)
print("Saved interactive map:", out)

## Proximity analysis
This map marks launchpads, not every launch trajectory. To make proximity claims, add a reliable geospatial dataset for coastlines, roads, railways, or cities and compute distances using a geographic method (for example, geodesic distance). Do not infer distances from the map display alone.